In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import random

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/F1Dataset/models"

model = load_model(f"{MODEL_PATH}/model.keras")
scaler = joblib.load(f"{MODEL_PATH}/scaler.pkl")
model_columns = joblib.load(f"{MODEL_PATH}/model_columns.pkl")
test_original = pd.read_csv(f"{MODEL_PATH}/test_original.csv")

print("Files loaded successfully.")
print("Test original shape:", test_original.shape)

Files loaded successfully.
Test original shape: (1271, 8)


In [ ]:
target_column = "positionOrder"

features = [
    "year",
    "round",
    "grid",
    "driverRef",
    "constructorRef",
    "circuitRef",
    "country"
]

In [ ]:
def predict_from_test_query(sample_index):
    sample = test_original.iloc[[sample_index]]

    query_features = sample[features]
    true_position = sample[target_column].values[0]

    query_encoded = pd.get_dummies(query_features)

    query_encoded = query_encoded.reindex(columns=model_columns, fill_value=0)

    query_scaled = scaler.transform(query_encoded)

    predicted_position = model.predict(query_scaled, verbose=0).flatten()[0]
    rounded_prediction = round(predicted_position)

    absolute_error = abs(true_position - predicted_position)
    rounded_error = abs(true_position - rounded_prediction)
    sep = "-" * 50

    print(sep)
    print("Input query:")
    display(query_features)

    print(sep)
    print("RESULTADO")
    print(sep)
    print(f"Prediccion del modelo {predicted_position:.2f}")
    print(f"Posicion redondeada   {rounded_prediction}")
    print(f"Posicion real         {true_position}")
    print(sep)
    print("ERROR")
    print(sep)
    print(f"  Error absoluto      {absolute_error:.2f} pos.")
    print(f"  Error redondeado    {rounded_error} pos.")
    print(sep)

In [ ]:
random_index = random.randint(0, len(test_original) - 1)
predict_from_test_query(sample_index=random_index)

--------------------------------------------------
Input query:


,year,round,grid,driverRef,constructorRef,circuitRef,country
1134,2014,7,7,alonso,ferrari,villeneuve,Canada


--------------------------------------------------
RESULTADO
--------------------------------------------------
Prediccion del modelo 7.11
Posicion redondeada   7
Posicion real         6
--------------------------------------------------
ERROR
--------------------------------------------------
  Error absoluto      1.11 pos.
  Error redondeado    1 pos.
--------------------------------------------------


In [ ]:
def predict_future_race(year, round_number, grid, driver, constructor, circuit, country):
    query = {
        "year": year,
        "round": round_number,
        "grid": grid,
        "driverRef": driver,
        "constructorRef": constructor,
        "circuitRef": circuit,
        "country": country
    }

    query_df = pd.DataFrame([query])

    query_encoded = pd.get_dummies(query_df)
    query_encoded = query_encoded.reindex(columns=model_columns, fill_value=0)

    query_scaled = scaler.transform(query_encoded)

    predicted_position = model.predict(query_scaled, verbose=0).flatten()[0]
    rounded_prediction = round(predicted_position)

    print("Future race query:")
    display(query_df)

    print("\nPrediction result:")
    print(f"Predicted position: {predicted_position:.2f}")
    print(f"Rounded prediction: {rounded_prediction}")

    return predicted_position

In [ ]:
year = int(input("Ingrese el año: "))
round_number = int(input("Ingrese el número de carrera: "))
grid = int(input("Ingrese la posición de inicio: "))
driver = input("Ingrese el piloto: ")
constructor = input("Ingrese la escudería: ")
circuit = input("Ingrese el nombre del circuito: ")
country = input("Ingrese el país del circuito: ")

predict_future_race(
    year=year,
    round_number=round_number,
    grid=grid,
    driver=driver,
    constructor=constructor,
    circuit=circuit,
    country=country
)

Ingrese el año: 2025
Ingrese el número de carrera: 3
Ingrese la posición de inicio: 4
Ingrese el piloto: verstappen
Ingrese la escudería: red_bull
Ingrese el nombre del circuito: shanghai
Ingrese el país del circuito: China
Future race query:


,year,round,grid,driverRef,constructorRef,circuitRef,country
0,2025,3,4,verstappen,red_bull,shanghai,China



Prediction result:
Predicted position: 4.06
Rounded prediction: 4


np.float32(4.0580626)

In [ ]:
predict_future_race(
    year=2025,
    round_number=3,
    grid=4,
    driver="verstappen",
    constructor="red_bull",
    circuit="shanghai",
    country="China"
)

Future race query:


,year,round,grid,driverRef,constructorRef,circuitRef,country
0,2025,3,4,verstappen,red_bull,shanghai,China



Prediction result:
Predicted position: 4.06
Rounded prediction: 4


np.float32(4.0580626)

Año 2024, número de carrera, inicia en la tercera posición, corredor checo perez, equipo red_bull, circuito jeddah,

In [ ]:
predict_future_race(
    year=2024,
    round_number=2,
    grid=3,
    driver="perez",
    constructor="red_bull",
    circuit="jeddah",
    country="Saudi Arabia"
)

Future race query:


,year,round,grid,driverRef,constructorRef,circuitRef,country
0,2024,2,3,perez,red_bull,jeddah,Saudi Arabia



Prediction result:
Predicted position: 6.07
Rounded prediction: 6


np.float32(6.072578)